In [ ]:
# === Optimized EdgeFormerNet++ Training Code for Dice > 0.85 === - TRAINING THE MODEL

import os
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.layers import (
    LayerNormalization, MultiHeadAttention, Dense, Dropout, Add, 
    GlobalAveragePooling2D, Multiply, Conv2D, Reshape, Concatenate, 
    BatchNormalization, Conv2DTranspose, Activation, ReLU, MaxPooling2D
)
from tensorflow.keras.utils import Sequence
import albumentations as A
from albumentations.core.composition import OneOf

# EdgeFormerNet++: OCT Layer Segmentation with Res2Net - FINAL MODEL
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from tensorflow.keras.layers import *
from tensorflow.keras.initializers import GlorotUniform

initializer = GlorotUniform(seed=42)

# === Edge Attention Block ===
def edge_attention_block(input_tensor):
    edge = tf.image.sobel_edges(input_tensor)
    edge = tf.reduce_sum(edge, axis=-1)
    edge = tf.reduce_mean(edge, axis=-1, keepdims=True)
    edge = Conv2D(16, (3, 3), padding='same', activation='relu')(edge)
    edge = Conv2D(1, (1, 1), activation='sigmoid')(edge)
    return Multiply()([input_tensor, edge])

# === CBAM Module ===
def cbam_block(x, ratio=8):
    channel = x.shape[-1]
    shared_dense = [
        Dense(channel // ratio, activation='relu'),
        Dense(channel)
    ]
    avg_pool = GlobalAveragePooling2D()(x)
    avg_pool = shared_dense[0](avg_pool)
    avg_pool = shared_dense[1](avg_pool)
    max_pool = GlobalMaxPooling2D()(x)
    max_pool = shared_dense[0](max_pool)
    max_pool = shared_dense[1](max_pool)
    cbam_feature = Add()([avg_pool, max_pool])
    cbam_feature = Activation('sigmoid')(cbam_feature)
    cbam_feature = Multiply()([x, cbam_feature])
    return cbam_feature

# === SCSE Module ===
def scse_block(x):
    channel = x.shape[-1]
    se = GlobalAveragePooling2D()(x)
    se = Dense(channel // 2, activation='relu')(se)
    se = Dense(channel, activation='sigmoid')(se)
    se = Multiply()([x, se])
    spatial = Conv2D(1, (1, 1), activation='sigmoid')(x)
    spatial = Multiply()([x, spatial])
    return Add()([se, spatial])

# === Transformer Block ===
def transformer_block(x, num_heads=4, ff_dim=128):
    x_norm = LayerNormalization()(x)
    attn_output = MultiHeadAttention(num_heads=num_heads, key_dim=ff_dim)(x_norm, x_norm)
    x = Add()([x, attn_output])
    ffn_output = Dense(ff_dim, activation='relu')(x)
    ffn_output = Dense(x.shape[-1])(ffn_output)
    return Add()([x, ffn_output])

# === Res2Net Block ===
def res2net_block(x, filters, scale=4):
    input_channels = x.shape[-1]
    split_channels = filters // scale
    convs = []
    for s in range(scale):
        if s == 0:
            conv = Conv2D(split_channels, (3, 3), padding='same', activation='relu')(x)
        else:
            prev = convs[-1]
            shortcut = Lambda(lambda z: z[:, :, :, :split_channels])(x) if input_channels >= split_channels else x
            added = Add()([prev, shortcut])
            conv = Conv2D(split_channels, (3, 3), padding='same', activation='relu')(added)
        convs.append(conv)
    out = Concatenate()(convs)
    out = Conv2D(filters, (1, 1), padding='same')(out)
    return out

# === Encoder using Res2Net ===
def encoder_res2net(input_tensor):
    skips = []
    x = Conv2D(32, (3, 3), strides=(1, 1), padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = ReLU()(x)

    for filters in [64, 128, 256, 512]:
        x = res2net_block(x, filters)
        skips.append(x)
        x = MaxPooling2D((2, 2))(x)
    return x, skips[::-1]  # reverse for decoder

# === Decoder Block ===
def decoder_block(x, skip, filters):
    x = Conv2DTranspose(filters, (3, 3), strides=(2, 2), padding='same')(x)
    x = Concatenate()([x, skip])
    x = Conv2D(filters, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    x = Conv2D(filters, (3, 3), padding='same')(x)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

# === Full Model ===
def build_edgeformernet(input_shape=(384, 256, 1), num_classes=9):
    inputs = Input(shape=input_shape)
    edge_feat = edge_attention_block(inputs)
    x = Concatenate()([inputs, edge_feat])

    x, skips = encoder_res2net(x)

    x = transformer_block(x)
    x = cbam_block(x)

    d4 = decoder_block(x, skips[0], 256)
    d4 = cbam_block(d4)
    d4 = scse_block(d4)

    d3 = decoder_block(d4, skips[1], 128)
    d2 = decoder_block(d3, skips[2], 64)
    d1 = decoder_block(d2, skips[3], 32)

    outputs = Conv2D(num_classes, (1, 1), activation='softmax')(d1)

    return models.Model(inputs, outputs, name='EdgeFormerNetPlus')

# === Example ===
model = build_edgeformernet()
model.summary()
# === CONFIG ===
img_height, img_width = 384, 256
batch_size = 4
#epochs = 50
epochs=400
#epochs = 500
initializer = tf.keras.initializers.GlorotUniform(seed=42)

# === Mask Color Mapping ===
def extract_unique_colors(mask_dir):
    unique_colors = set()
    for fname in os.listdir(mask_dir):
        if fname.endswith(('.png', '.bmp')):
            mask = np.array(Image.open(os.path.join(mask_dir, fname)).convert('RGB'))
            colors = np.unique(mask.reshape(-1, 3), axis=0)
            for c in map(tuple, colors):
                unique_colors.add(c)
    return sorted(list(unique_colors))

train_lbl_dir = r"D:\layer\NR206\train_labels"
UNIQUE_COLORS = extract_unique_colors(train_lbl_dir)
COLOR_TO_INDEX = {color: idx for idx, color in enumerate(UNIQUE_COLORS)}
INDEX_TO_COLOR = {idx: color for color, idx in COLOR_TO_INDEX.items()}
num_classes = len(COLOR_TO_INDEX)

# === Load Image/Mask Pair ===
def load_image_mask_pair(image_path, mask_path):
    image = Image.open(image_path).resize((img_width, img_height))
    image = np.array(image, dtype=np.float32) / 255.0
    if image.ndim == 2:
        image = np.stack([image] * 3, axis=-1)

    mask = Image.open(mask_path).convert('RGB').resize((img_width, img_height), resample=Image.NEAREST)
    mask = np.array(mask)
    mask_indexed = np.zeros((img_height, img_width), dtype=np.uint8)
    for color, idx in COLOR_TO_INDEX.items():
        mask_indexed[np.all(mask == color, axis=-1)] = idx
    return image, mask_indexed

# === Dataset Loader ===
def load_dataset(image_dir, mask_dir):
    image_files = sorted([os.path.join(image_dir, f) for f in os.listdir(image_dir) if f.endswith(('.png', '.bmp'))])
    mask_files = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir) if f.endswith(('.png', '.bmp'))])
    images, masks = [], []
    for img_path, msk_path in zip(image_files, mask_files):
        img, msk = load_image_mask_pair(img_path, msk_path)
        images.append(img)
        masks.append(msk)
    return np.array(images), np.array(masks)

# === Augmentation ===
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ElasticTransform(p=0.4),
    A.GridDistortion(p=0.2),
    A.RandomBrightnessContrast(p=0.4),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.4),
    A.CLAHE(p=0.2),
    OneOf([
        A.GaussianBlur(blur_limit=(3, 5)),
        A.MedianBlur(blur_limit=5),
        A.GaussNoise(var_limit=(10.0, 50.0))
    ], p=0.3)
])

# === Custom Data Generator ===
class CustomDataGenerator(Sequence):
    def __init__(self, images, masks, batch_size, transform=None):
        self.images = images
        self.masks = masks
        self.batch_size = batch_size
        self.transform = transform

    def __len__(self):
        return len(self.images) // self.batch_size

    def __getitem__(self, idx):
        batch_images = self.images[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_masks = self.masks[idx * self.batch_size:(idx + 1) * self.batch_size]
        aug_imgs, aug_masks = [], []
        for img, mask in zip(batch_images, batch_masks):
            augmented = self.transform(image=img, mask=mask) if self.transform else {'image': img, 'mask': mask}
            aug_imgs.append(np.resize(augmented['image'], (img_height, img_width, 3)).astype(np.float32))
            aug_masks.append(np.resize(augmented['mask'], (img_height, img_width)).astype(np.uint8))
        return np.stack(aug_imgs), np.stack(aug_masks)

# === Loss Functions ===
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2])
    union = tf.reduce_sum(y_true + y_pred, axis=[1, 2])
    return 1.0 - tf.reduce_mean((2. * intersection + smooth) / (union + smooth))

def tversky_loss(y_true, y_pred, alpha=0.6, beta=0.4, smooth=1e-6):
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0)
    TP = tf.reduce_sum(y_true * y_pred, axis=[1,2])
    FP = tf.reduce_sum((1 - y_true) * y_pred, axis=[1,2])
    FN = tf.reduce_sum(y_true * (1 - y_pred), axis=[1,2])
    return tf.reduce_mean(1 - (TP + smooth) / (TP + alpha * FP + beta * FN + smooth))

def hybrid_loss(y_true, y_pred):
    return 0.5 * dice_loss(y_true, y_pred) + 0.5 * tversky_loss(y_true, y_pred)

def dice_metric(y_true, y_pred, smooth=1e-6):
    y_pred = tf.argmax(y_pred, axis=-1)
    y_pred = tf.one_hot(y_pred, depth=num_classes)
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2])
    union = tf.reduce_sum(y_true + y_pred, axis=[1, 2])
    return tf.reduce_mean((2. * intersection + smooth) / (union + smooth))

def iou_metric(y_true, y_pred, smooth=1e-6):
    y_pred = tf.argmax(y_pred, axis=-1)
    y_pred = tf.one_hot(y_pred, depth=num_classes)
    y_true = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2])
    union = tf.reduce_sum(y_true + y_pred, axis=[1, 2]) - intersection
    return tf.reduce_mean((intersection + smooth) / (union + smooth))

# === Load Data ===
train_img_dir = r"D:\layer\NR206\train"
val_img_dir = r"D:\layer\NR206\val"
val_lbl_dir = r"D:\layer\NR206\val_labels"
X_train, y_train = load_dataset(train_img_dir, train_lbl_dir)
X_val, y_val = load_dataset(val_img_dir, val_lbl_dir)
train_gen = CustomDataGenerator(X_train, y_train, batch_size=batch_size, transform=train_transform)
val_gen = CustomDataGenerator(X_val, y_val, batch_size=batch_size)

# === Build Model ===
model = build_edgeformernet(input_shape=(img_height, img_width, 3), num_classes=num_classes)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss=hybrid_loss,
              metrics=[dice_metric, iou_metric])

# === Training Callbacks ===
callbacks = [
    #tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=30, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint("EdgeFormer_Dice85_best_val_loss_{val_loss:.6f}.h5", save_best_only=True, monitor='val_loss'),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=30, min_lr=1e-6)
]

# === Train ===
history = model.fit(train_gen, validation_data=val_gen, epochs=epochs, callbacks=callbacks)


C:\Users\VPG\.conda\envs\micaih_clean\lib\site-packages\albumentations\__init__.py:13: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.18). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Model: "EdgeFormerNetPlus"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 384, 256, 1  0           []                               
                                )]                                                                
                                                                                                  
 tf.compat.v1.shape (TFOpLambda  (4,)                0           ['input_1[0][0]']                
 )                                                                                                
                                                                                                  
 tf.__operators__.getitem (Slic  ()                  0           ['tf.compat.v1.shape[0][0]']     
 ingOpLambda)                                                                     